# Search-5-GeneticAlgorithms-Csharp : Algorithmes Génétiques en C#

**Navigation** : [<< Search-4 (Local Search)](Search-4-LocalSearch.ipynb) | [↑ Série Search (C#)](../README.md) | [Search-6 (Adversarial Search) >>](Search-6-AdversarialSearch.ipynb)

**Jumeau .NET** du notebook Python [Search-5-GeneticAlgorithms](Search-5-GeneticAlgorithms.ipynb). Port C# from-scratch (EPIC [#4956](https://github.com/jsboige/CoursIA/issues/4956), prong A [#3801](https://github.com/jsboige/CoursIA/issues/3801) — algorithme pur, aucun framework GA externe : .NET n'ayant pas de framework GA dominant, l'implémentation from-scratch **est** la leçon). Tables texte via `.Display()` (verdict #3436 INTRINSIC : path-leaks SkiaSharp intrinsèques au kernel .NET Interactive).

## Objectifs d'apprentissage

1. **Encoder** un chromosome (binaire OneMax, réel Rastrigin) en C#
2. **Implémenter** sélection (roulette, tournoi), crossover (1-point, arithmétique), mutation (bit-flip, gaussienne)
3. **Assembler** une classe `GeneticAlgorithm<TChromosome>` générique réutilisable
4. **Étudier** la convergence sur OneMax + l'optimisation continue sur Rastrigin
5. **Mesurer** l'effet de l'élitisme et d'un taux de mutation adaptatif

### Prérequis

- .NET 9.0 + kernel `.net-csharp`
- Search-4-LocalSearch (notion de paysage d'optimisation)

### Durée estimée : ~1h

## 1. Introduction (~5 min)

Un **algorithme génétique (GA)** est une métaheuristique d'optimisation inspirée de l'évolution naturelle. Une **population** de **chromosomes** (solutions candidates) évolue sur plusieurs **générations** via :

1. **Sélection** — les individus les plus *aptes* (fitness élevé) ont plus de descendants.
2. **Crossover** (croisement) — deux parents produisent des enfants combinant leurs gènes.
3. **Mutation** — perturbation aléatoire, source de diversité et d'exploration.

**Pourquoi les AG ?** Ils excellent sur les espaces de recherche vastes, non-différentiables, multimodaux — là où le gradient est inutilisable (Search-4 a montré les limites du hill-climbing sur les paysages accidentés).

## 2. Composants d'un AG (~10 min)

### 2.1 Encodage du chromosome

Le choix de l'encodage conditionne tout. Deux cas canoniques :

| Problème | Encodage | Chromosome C# |
|----------|----------|---------------|
| OneMax (maximiser la somme de bits) | binaire | `bool[]` |
| Rastrigin (minimiser f(x)) | réel | `double[]` |

### 2.2 Opérateurs

Classe GA **générique** paramétrée par le type de gène : `record` immutable pour l'individu + délégués pour chaque opérateur (pattern établi par les jumeaux Search-6/13/14-Csharp).

In [1]:
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Linq;
using System.Text;

// Individu : chromosome + fitness calculé
public record Individu<T>(T Chromosome, double Fitness);

// GA générique : l'utilisateur fournit fitness + 4 opérateurs (délégués)
public class GeneticAlgorithm<T>
{
    private readonly Random _rng;
    private readonly Func<T,double> _fitness;
    private readonly Func<Random,T> _randomIndiv;
    private readonly Func<Random,T,T,(T,T)> _crossover;
    private readonly Func<Random,T,double,T> _mutate;
    private readonly Func<Random,Individu<T>[],int> _select;
    public int PopulationSize { get; }
    public double MutationRate { get; set; }
    public bool Elitism { get; set; }
    public int EliteCount { get; set; } = 1;
    public List<double> HistoryBest { get; } = new();
    public List<double> HistoryAvg { get; } = new();

    public GeneticAlgorithm(Func<T,double> fitness, Func<Random,T> randomIndiv,
        Func<Random,T,T,(T,T)> crossover, Func<Random,T,double,T> mutate,
        Func<Random,Individu<T>[],int> select,
        int populationSize, double mutationRate, int seed = 42)
    { _rng=new(seed); _fitness=fitness; _randomIndiv=randomIndiv; _crossover=crossover;
      _mutate=mutate; _select=select; PopulationSize=populationSize; MutationRate=mutationRate; }

    public Individu<T>[] Run(int generations)
    {
        var pop = Enumerable.Range(0, PopulationSize).Select(_=>Eval(_randomIndiv(_rng))).ToArray();
        Record(pop);
        for (int g=0; g<generations; g++) {
            var next = new List<Individu<T>>();
            if (Elitism) next.AddRange(pop.OrderByDescending(p=>p.Fitness).Take(EliteCount));
            while (next.Count < PopulationSize) {
                int i1=_select(_rng,pop), i2=_select(_rng,pop);
                var (c1,c2)=_crossover(_rng, pop[i1].Chromosome, pop[i2].Chromosome);
                next.Add(Eval(_mutate(_rng, c1, MutationRate)));
                if (next.Count<PopulationSize) next.Add(Eval(_mutate(_rng, c2, MutationRate)));
            }
            pop = next.ToArray(); Record(pop);
        }
        return pop;
    }
    private Individu<T> Eval(T chrom) => new(chrom, _fitness(chrom));
    private void Record(Individu<T>[] pop) { HistoryBest.Add(pop.Max(p=>p.Fitness)); HistoryAvg.Add(pop.Average(p=>p.Fitness)); }
}

var sb = new StringBuilder();
sb.AppendLine("Classe GeneticAlgorithm<T> chargée — opérateurs (sélection/crossover/mutation) = délégués.");
sb.ToString().Display();

Classe GeneticAlgorithm<T> chargée — opérateurs (sélection/crossover/mutation) = délégués.


#### Opérateurs binaires (OneMax)

- **Sélection roulette** : probabilité proportionnelle au fitness ; **tournoi** : meilleur de k tirés au hasard.
- **Crossover 1-point** : un point de coupure aléatoire.
- **Mutation bit-flip** : chaque bit inversé avec probabilité `mutationRate`.

In [2]:
// Opérateurs binaire (OneMax)
double OneMax(bool[] c) => c.Count(b=>b);
bool[] RandomBits(Random r) => Enumerable.Range(0,20).Select(_=>r.NextDouble()<0.5).ToArray();
(bool[],bool[]) Crossover1P(Random r, bool[] a, bool[] b) {
    int pt = r.Next(1, a.Length);
    return (a.Take(pt).Concat(b.Skip(pt)).ToArray(), b.Take(pt).Concat(a.Skip(pt)).ToArray());
}
bool[] MutateBitFlip(Random r, bool[] c, double rate) {
    var m=(bool[])c.Clone();
    for (int i=0;i<m.Length;i++) if (r.NextDouble()<rate) m[i]=!m[i];
    return m;
}
int Roulette(Random r, Individu<bool[]>[] pop) {
    double min=pop.Min(p=>p.Fitness);
    double[] w=pop.Select(p=>p.Fitness-min+1e-9).ToArray();
    double total=w.Sum(), pick=r.NextDouble()*total, acc=0;
    for (int i=0;i<w.Length;i++){ acc+=w[i]; if(acc>=pick) return i; }
    return w.Length-1;
}
int Tournament(Random r, Individu<bool[]>[] pop, int k=3) {
    int best=r.Next(pop.Length);
    for (int i=1;i<k;i++){ int c=r.Next(pop.Length); if(pop[c].Fitness>pop[best].Fitness) best=c; }
    return best;
}

var sb = new StringBuilder();
sb.AppendLine("Opérateurs binaires (OneMax) chargés :");
sb.AppendLine("  OneMax, RandomBits(L=20), Crossover1P, MutateBitFlip, Roulette, Tournament(k=3)");
sb.ToString().Display();

Opérateurs binaires (OneMax) chargés :
  OneMax, RandomBits(L=20), Crossover1P, MutateBitFlip, Roulette, Tournament(k=3)


## 3. AG from-scratch : OneMax (~10 min)

**OneMax** = maximiser la somme des bits d'un chromosome binaire. Solution optimale triviale (tous les bits à 1), ce qui rend la **convergence** (et non le résultat) l'objet d'étude : un GA sain converge en un nombre de générations prévisible.

In [3]:
// OneMax : exécution + convergence (table texte)
var ga = new GeneticAlgorithm<bool[]>(OneMax, RandomBits, Crossover1P, MutateBitFlip, Roulette,
            populationSize:100, mutationRate:0.01, seed:42);
var sw = Stopwatch.StartNew();
var final = ga.Run(generations:40);
sw.Stop();
var best = final.OrderByDescending(p=>p.Fitness).First();

var sb = new StringBuilder();
sb.AppendLine("=== OneMax GA (pop=100, 40 générations, taux mutation 0.01) ===");
sb.AppendLine($"Meilleur fitness final : {best.Fitness}/20");
sb.AppendLine($"Temps : {sw.Elapsed.TotalMilliseconds:F1} ms");
sb.AppendLine();
sb.AppendLine("Génération | Best | Moyenne");
sb.AppendLine("-----------|------|--------");
foreach (var g in new[]{0,5,10,20,30,40})
    sb.AppendLine($"{g,10} | {ga.HistoryBest[g],4:F0} | {ga.HistoryAvg[g],6:F2}");
sb.AppendLine();
if (best.Fitness >= 19) sb.AppendLine("Convergence : OneMax résolu (>= 19/20). GA from-scratch fonctionnel.");
else sb.AppendLine($"Convergence partielle : {best.Fitness}/20 (augmenter générations).");
sb.ToString().Display();

=== OneMax GA (pop=100, 40 générations, taux mutation 0.01) ===
Meilleur fitness final : 20/20
Temps : 98,2 ms

Génération | Best | Moyenne
-----------|------|--------
         0 |   15 |  10,20
         5 |   17 |  13,55
        10 |   19 |  16,41
        20 |   20 |  19,17
        30 |   20 |  19,55
        40 |   20 |  19,72

Convergence : OneMax résolu (>= 19/20). GA from-scratch fonctionnel.


## 4. Optimisation continue : Rastrigin (~8 min)

**Rastrigin** est la fonction-test classique : fortement multimodale (des milliers d'optima locaux), minimum global 0 en x=0. C'est l'anti-OneMax : le gradient mène aux pièges locaux, le GA doit donc **explorer**.

$$f(\mathbf{x}) = 10d + \sum_{i=1}^{d} \left[ x_i^2 - 10\cos(2\pi x_i) \right]$$

On réutilise **la même classe `GeneticAlgorithm<T>`** avec des opérateurs réels (crossover arithmétique + mutation gaussienne). Le Python appelle ça DEAP (§4) ou PyGAD (§5) ; en .NET on le code en ~20 lignes, ce qui **démystifie** le framework.

In [4]:
// Rastrigin : opérateurs réels + exécution
double Rastrigin(double[] x) {
    double s = 10.0*x.Length;
    for (int i=0;i<x.Length;i++) s += x[i]*x[i] - 10.0*Math.Cos(2*Math.PI*x[i]);
    return s;
}
double NegRastrigin(double[] x) => -Rastrigin(x);
double[] RandomReal(Random r) => Enumerable.Range(0,5).Select(_=>r.NextDouble()*10.24-5.12).ToArray();
(double[],double[]) CrossoverArith(Random r, double[] a, double[] b) {
    double alpha=r.NextDouble();
    return (a.Zip(b,(x,y)=>alpha*x+(1-alpha)*y).ToArray(), a.Zip(b,(x,y)=>(1-alpha)*x+alpha*y).ToArray());
}
double[] MutateGauss(Random r, double[] c, double rate, double sigma=0.5) {
    var m=(double[])c.Clone();
    for (int i=0;i<m.Length;i++) if (r.NextDouble()<rate)
        m[i] += sigma*(r.NextDouble()+r.NextDouble()+r.NextDouble()-1.5);
    for (int i=0;i<m.Length;i++) m[i]=Math.Max(-5.12, Math.Min(5.12, m[i]));
    return m;
}
int RouletteReal(Random r, Individu<double[]>[] pop) {
    double min=pop.Min(p=>p.Fitness);
    double[] w=pop.Select(p=>p.Fitness-min+1e-9).ToArray();
    double total=w.Sum(), pick=r.NextDouble()*total, acc=0;
    for (int i=0;i<w.Length;i++){ acc+=w[i]; if(acc>=pick) return i; }
    return w.Length-1;
}

var gaR = new GeneticAlgorithm<double[]>(NegRastrigin, RandomReal, CrossoverArith,
            (rng,c,rate)=>MutateGauss(rng,c,rate,0.5), RouletteReal,
            populationSize:200, mutationRate:0.1, seed:7);
var swR = Stopwatch.StartNew();
var finalR = gaR.Run(generations:100);
swR.Stop();
var bestR = finalR.OrderByDescending(p=>p.Fitness).First();
double fRastrigin = Rastrigin(bestR.Chromosome);

var sb = new StringBuilder();
sb.AppendLine("=== Rastrigin GA (d=5, pop=200, 100 générations) ===");
sb.AppendLine($"Meilleur f(x) trouvé : {fRastrigin:F4}  (optimum global = 0.0000)");
sb.AppendLine($"Temps : {swR.Elapsed.TotalMilliseconds:F1} ms");
sb.AppendLine();
sb.AppendLine("Génération | f(x) meilleur | Moyenne fitness");
sb.AppendLine("-----------|---------------|----------------");
foreach (var g in new[]{0,20,40,60,80,100})
    sb.AppendLine($"{g,10} | {-gaR.HistoryBest[g],13:F4} | {gaR.HistoryAvg[g],14:F2}");
sb.AppendLine();
string verdict = fRastrigin<1.0 ? "Très proche de l'optimum global (< 1.0)." :
                 fRastrigin<5.0 ? "Bon résultat (< 5.0) — optimum local évité." :
                                  "Piégé dans un optimum local (Rastrigin est dur).";
sb.AppendLine($"Verdict : {verdict}");
sb.AppendLine("Rastrigin illustre le rôle-clé de la MUTATION (exploration) vs OneMax (exploitation).");
sb.ToString().Display();

=== Rastrigin GA (d=5, pop=200, 100 générations) ===
Meilleur f(x) trouvé : 1,2566  (optimum global = 0.0000)
Temps : 507,1 ms

Génération | f(x) meilleur | Moyenne fitness
-----------|---------------|----------------
         0 |       25,6282 |         -91,27
        20 |        1,9352 |         -19,89
        40 |        2,2628 |         -13,15
        60 |        1,3856 |         -10,61
        80 |        1,2280 |          -8,41
       100 |        1,2566 |          -9,39

Verdict : Bon résultat (< 5.0) — optimum local évité.
Rastrigin illustre le rôle-clé de la MUTATION (exploration) vs OneMax (exploitation).


## 5. Concepts avancés (~6 min)

### 5.1 Élitisme

L'**élitisme** préserve les N meilleurs individus d'une génération à la suivante. Il garantit que le meilleur jamais trouvé ne se dégrade jamais — au prix d'un risque accru de convergence prématurée.

In [5]:
// Effet de l'élitisme sur OneMax
var sb = new StringBuilder();
sb.AppendLine("=== Élitisme : avec vs sans (OneMax, 30 générations) ===");
sb.AppendLine();
sb.AppendLine("Config        | Best final | Génération convergence (>=19)");
sb.AppendLine("-------------|------------|------------------------------");
foreach (var (elite, label) in new[] { (false,"Sans élite  "), (true,"Avec élite ") }) {
    int convGen = -1;
    var g = new GeneticAlgorithm<bool[]>(OneMax, RandomBits, Crossover1P, MutateBitFlip, Roulette,
              populationSize:100, mutationRate:0.01, seed:42){ Elitism=elite, EliteCount=1 };
    g.Run(30);
    for (int i=0;i<g.HistoryBest.Count;i++) if (g.HistoryBest[i]>=19){ convGen=i; break; }
    sb.AppendLine($"{label} | {g.HistoryBest.Last(),10:F0} | {convGen}");
}
sb.AppendLine();
sb.AppendLine("L'élitisme accélère la convergence mais peut réduire la diversité.");
sb.AppendLine("Compromis exploration/exploitation : pas de 'bon' réglage universel (No-Free-Lunch).");
sb.ToString().Display();

=== Élitisme : avec vs sans (OneMax, 30 générations) ===

Config        | Best final | Génération convergence (>=19)
-------------|------------|------------------------------
Sans élite   |         20 | 3
Avec élite  |         20 | 4

L'élitisme accélère la convergence mais peut réduire la diversité.
Compromis exploration/exploitation : pas de 'bon' réglage universel (No-Free-Lunch).


### 5.2 Taux de mutation adaptatif

Plutôt qu'un taux fixe, on le fait **décroître** avec les générations : forte exploration au début, raffinement à la fin.

In [6]:
// Taux de mutation adaptatif : décroissance linéaire
double AdaptiveRate(int gen, int maxGen, double rMax=0.1, double rMin=0.001) {
    double t = (double)gen/maxGen;
    return rMax - (rMax-rMin)*t;
}
var sb = new StringBuilder();
sb.AppendLine("=== Taux de mutation adaptatif (décroissance linéaire) ===");
sb.AppendLine();
sb.AppendLine("Génération | Taux mutation");
sb.AppendLine("-----------|--------------");
foreach (var g in new[]{0,25,50,75,100})
    sb.AppendLine($"{g,10} | {AdaptiveRate(g,100),12:F4}");
sb.AppendLine();
sb.AppendLine("Intuition : on explore largement au début (~0.1), on raffine à la fin (~0.001).");
sb.AppendLine("Exercice 3 : branchez AdaptiveRate dans GeneticAlgorithm.Run (voir ci-dessous).");
sb.ToString().Display();

=== Taux de mutation adaptatif (décroissance linéaire) ===

Génération | Taux mutation
-----------|--------------
         0 |       0,1000
        25 |       0,0753
        50 |       0,0505
        75 |       0,0257
       100 |       0,0010

Intuition : on explore largement au début (~0.1), on raffine à la fin (~0.001).
Exercice 3 : branchez AdaptiveRate dans GeneticAlgorithm.Run (voir ci-dessous).


## Synthèse et lien avec les Applications

Les GA de cette feuille sont la **brique** derrière plusieurs notebooks d'application :
- **App-13 TSP-Metaheuristiques** : GA + 2-opt sur le voyageur de commerce.
- **App-17 VRP-Logistics** : optimisation de tournées de livraison.
- **App-18 HyperparameterTuning** : GA pour explorer l'espace d'hyperparamètres.

Le pattern `GeneticAlgorithm<T>` générique est conçu pour être **réutilisé** : il suffit de fournir fitness + 4 opérateurs adaptés au problème.

## Exercices

**Exercice 1 — Tournoi vs roulette.** Remplacez la sélection roulette par le tournoi (`Tournament`, k=3) sur OneMax. Comparez la génération de convergence (>=19). Quelle sélection converge plus vite ? Pourquoi ?

**Exercice 2 — Effet de la taille de population.** Faites varier `PopulationSize` ∈ {20, 50, 100, 200} sur Rastrigin (mêmes générations). Affichez (table texte) le meilleur f(x) final. Y a-t-il un rendement décroissant ?

**Exercice 3 — Mutation adaptative.** Modifiez `GeneticAlgorithm<T>.Run` pour que le taux de mutation passe à `AdaptiveRate(g, generations)` à chaque génération. Relancez OneMax. Comparez la trajectoire de convergence au taux fixe 0.01.

In [7]:
// Exercice 1 — Tournoi vs roulette (étudiant à compléter)
// Remplacez la sélection Roulette par Tournament (k=3) sur OneMax, mesurez convGen.
"Exercice 1 à compléter : comparer Tournament vs Roulette (génération de convergence >= 19).".Display();


Exercice 1 à compléter : comparer Tournament vs Roulette (génération de convergence >= 19).

In [8]:
// Exercice 2 — Effet de la taille de population (étudiant à compléter)
// Bouclez sur PopulationSize in {20, 50, 100, 200} sur Rastrigin, collectez best f(x).
"Exercice 2 à compléter : sweep PopulationSize sur Rastrigin, afficher best f(x) final.".Display();


Exercice 2 à compléter : sweep PopulationSize sur Rastrigin, afficher best f(x) final.

In [9]:
// Exercice 3 — Mutation adaptative (étudiant à compléter)
// Surchagez Run pour appeler MutateGauss avec AdaptiveRate(g, generations), comparez la trajectoire.
"Exercice 3 à compléter : brancher AdaptiveRate dans Run, comparer la convergence au taux fixe.".Display();


Exercice 3 à compléter : brancher AdaptiveRate dans Run, comparer la convergence au taux fixe.

---

## Conclusion

Vous avez implémenté **from-scratch** un algorithme génétique générique en C#, validé sur deux problèmes-test opposés (OneMax binaire, Rastrigin continu), et étudié deux leviers-clés (élitisme, mutation adaptatif). Leçon centrale : un GA n'est pas une boîte noire — c'est **sélection + crossover + mutation** assemblés sur une population, et .NET n'a pas besoin de framework GA pour l'enseigner.

**Navigation** : [<< Search-4 Local Search](Search-4-LocalSearch.ipynb) | [↑ Série Search (C#)](../README.md) | [Search-6 Adversarial Search >>](Search-6-AdversarialSearch.ipynb)

### Références

- Holland, J.H. (1975). *Adaptation in Natural and Artificial Systems*.
- Goldberg, D.E. (1989). *Genetic Algorithms in Search, Optimization and Machine Learning*.
- Jumeau Python : [Search-5-GeneticAlgorithms](Search-5-GeneticAlgorithms.ipynb) (DEAP + PyGAD).